# Avance Fase 3 — Semana 2

## Configuración inicial del proyecto

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO
# =====================================================================

# Ruta escrita desde la raíz del proyecto, no desde la carpeta de este cuaderno
RUTA_DATOS = "data/processed/listings_procesado.csv"

COLUMNA_OBJETIVO = "price_clp"       # la variable que se quiere explicar
COLUMNA_ID = "id"                    # identificador: se elimina del análisis

COLUMNAS_CONTINUAS = [
    # ubicación
    "latitude", "longitude",
    # tamaño y capacidad
    "accommodates", "bedrooms", "beds", "banos_numero", "amenities_count",
    # reglas de estadía
    "minimum_nights", "maximum_nights",
    # disponibilidad y demanda
    "availability_30", "availability_60", "availability_90", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d",
    "reviews_per_month", "estimated_occupancy_l365d", "antiguedad_dias",
    # puntajes de reseñas
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication",
    "review_scores_location", "review_scores_value",
    # anfitrión
    "calculated_host_listings_count",
]

COLUMNAS_NOMINALES = ["neighbourhood_cleansed", "property_type", "propiedad_agrupada",
                      "host_is_superhost"]   # superhost viene como texto t/f

COLUMNAS_BINARIAS = [
    "bedrooms_imputado", "beds_imputado", "ocupacion_tope", "es_multianfitrion",
    "amen_kitchen", "amen_wifi", "amen_hot_water", "amen_hair_dryer",
    "amen_cooking_basics", "amen_dishes_and_silverware", "amen_hangers",
    "amen_bed_linens", "amen_tv", "amen_microwave", "amen_self_check_in",
    "amen_refrigerator", "amen_iron", "amen_hot_water_kettle", "amen_wine_glasses",
    "amen_elevator", "amen_extra_pillows_and_blankets", "amen_long_term_stays_allowed",
    "amen_oven", "amen_toaster",
]

# Columnas del archivo que se dejan fuera a propósito (ver notas abajo)
COLUMNAS_EXCLUIDAS = ["host_id", "room_type", "banos_compartidos", "first_review",
                      "last_review", "precio_por_huesped", "estimated_revenue_l365d"]

SEMILLA = 42
PROPORCION_PRUEBA = 0.2

# Datos de demostración: solo para revisar el cuaderno sin el archivo real.
# Con True, los resultados NO son los de su proyecto. En la entrega debe ir False.
PERMITIR_DEMOSTRACION = False
# =====================================================================

COLUMNAS_ESPERADAS = ([COLUMNA_ID, COLUMNA_OBJETIVO] + COLUMNAS_CONTINUAS
                      + COLUMNAS_NOMINALES + COLUMNAS_BINARIAS)

from pathlib import Path


def localizar_raiz(inicio=None):
    """Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde está guardado el cuaderno (F3), no en la
    carpeta desde la que se lanzó JupyterLab. Si la carpeta de trabajo es una carpeta
    de fase, la raíz es la carpeta superior.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()
RUTA_DATOS = str(RAIZ / RUTA_DATOS)
DIR_PROCESADO = str(RAIZ / "data" / "processed")   # salidas compartidas por las fases

print("Raíz del proyecto   :", RAIZ.resolve())
print("Archivo configurado :", RUTA_DATOS)
print("Variable objetivo   :", COLUMNA_OBJETIVO)
print("Columnas esperadas  :", len(COLUMNAS_ESPERADAS))

Raíz del proyecto   : /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500
Archivo configurado : /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/data/processed/listings_procesado.csv
Variable objetivo   : price_clp
Columnas esperadas  : 57


## Preparación del entorno

In [2]:
import os
import sys
import time
import timeit
import tracemalloc

import numpy as np
import pandas as pd

SEMILLA = 42
np.random.seed(SEMILLA)

print("Python :", sys.executable)
print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Semilla:", SEMILLA)

Python : /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/.venv/bin/python
pandas : 3.1.1
NumPy  : 2.5.4
Semilla: 42


## Carga del data set

**Se realiza una carga de datos para verificar que se este leyendo correctamente desde la carpeta externa en la que se encuentra**

In [3]:
# Las funciones de carga se movieron a src/carga.py para reutilizarlas
# desde cualquier fase. El cuaderno las importa en vez de definirlas.
DIR_SRC = RAIZ / "src"
if str(DIR_SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(DIR_SRC.resolve()))

from carga import cargar

datos = cargar(RUTA_DATOS, COLUMNAS_ESPERADAS, COLUMNAS_CONTINUAS)
datos.head()


Forma: 3313 filas x 64 columnas
Esquema verificado: 57 columnas declaradas, 7 no declaradas
  No declaradas: ['host_id', 'room_type', 'banos_compartidos', 'precio_por_huesped', 'estimated_revenue_l365d', 'first_review', 'last_review']


,id,host_id,neighbourhood_cleansed,latitude,longitude,property_type,propiedad_agrupada,room_type,accommodates,bedrooms,...,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,host_is_superhost,calculated_host_listings_count,es_multianfitrion
0,1226526,828029c6c29f,Providencia,-33.42681,-70.60961,Entire rental unit,Entire rental unit,Entire home/apt,4,2.0,...,4.67,4.74,4.83,4.91,4.92,4.94,4.72,t,2,1
1,1300359,250fdbee5201,Providencia,-33.43005,-70.61772,Entire rental unit,Entire rental unit,Entire home/apt,4,2.0,...,4.73,4.71,4.80,4.79,4.85,4.85,4.61,f,20,1
2,3735992,481459247d1a,Estación Central,-33.45593,-70.70020,Entire condo,Entire condo,Entire home/apt,5,2.0,...,4.71,4.81,4.69,4.76,4.90,4.28,4.62,t,9,1
3,3810101,9e622c2063be,Lo Barnechea,-33.35603,-70.24830,Entire rental unit,Entire rental unit,Entire home/apt,5,2.0,...,4.91,4.89,4.71,4.89,4.86,4.89,4.77,f,1,0
4,1446357,476b08a4a2a1,Lo Barnechea,-33.34869,-70.29280,Entire condo,Entire condo,Entire home/apt,6,2.0,...,4.68,4.64,4.51,4.67,4.89,4.83,4.42,f,1,0


**Se perfilan los datos del data set**

In [4]:
def perfilar(df):
    """Devuelve una fila por columna con su tipo, únicos y porcentaje de nulos."""
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).sort_values("pct_nulos", ascending=False).reset_index(drop=True)


perfil = perfilar(datos)
faltantes = perfil[perfil["nulos"] > 0]
print("Columnas con valores faltantes:")
if faltantes.empty:
    print("  Ninguna: el conjunto de datos no tiene valores faltantes.")
else:
    print(faltantes.to_string(index=False))

# La variable objetivo es continua: se describe con estadísticos, no con frecuencias
print(f"\nDistribución de la variable objetivo '{COLUMNA_OBJETIVO}':")
print(datos[COLUMNA_OBJETIVO].describe().round(2))
print(f"Asimetría (skew)          : {datos[COLUMNA_OBJETIVO].skew():.2f}")
print(f"Asimetría tras log1p      : {np.log1p(datos[COLUMNA_OBJETIVO]).skew():.2f}")

print("\nCategorías de property_type (ordenadas por frecuencia):")
print(datos["property_type"].value_counts())

print("\nCategorías de propiedad_agrupada:")
print(datos["propiedad_agrupada"].value_counts())

Columnas con valores faltantes:
  Ninguna: el conjunto de datos no tiene valores faltantes.

Distribución de la variable objetivo 'price_clp':
count        3313.00
mean        98274.27
std        650820.17
min         19125.00
25%         44760.00
50%         59950.00
75%         85500.00
max      37034021.50
Name: price_clp, dtype: float64
Asimetría (skew)          : 55.25
Asimetría tras log1p      : 1.83

Categorías de property_type (ordenadas por frecuencia):
property_type
Entire rental unit           2926
Entire condo                  168
Entire serviced apartment      76
Entire home                    66
Entire loft                    30
Entire vacation home            9
Entire cabin                    8
Entire guesthouse               8
Tiny home                       7
Entire guest suite              4
Entire chalet                   3
Entire villa                    2
Camper/RV                       1
Entire cottage                  1
Casa particular                 1
Windmill 

## De las funciones de la Fase 2 a clases que aprenden

La Fase 3 no rehace el preprocesamiento: toma los pasos de F2 que **aprenden
del dataset** (los que calculan un parámetro con los datos y después lo
aplican) y los convierte en clases. El resto del pipeline sigue siendo
funciones, porque no memoriza nada.

Los tres pasos elegidos son los que más aportan al análisis:

- `beds` y `bedrooms`: sus nulos se imputan con la mediana del tramo de
  `accommodates` (decisión de F2, estrategia mediana por grupo).
- `property_type`: se agrupa en las 5 categorías más frecuentes + "Otro".
- `amenities`: se binarizan las 20 comodidades más frecuentes + conteo.

Cada clase sigue el mismo contrato: `ajustar(df)` aprende los parámetros y
`transformar(df)` los aplica. El estado aprendido vive en `_parametros`
(interno, por convención) y no se puede transformar antes de ajustar.


In [5]:
from transformadores import (ImputadorPorGrupo, AgrupadorTopCategorias,
                          BinarizadorAmenidades, parsear_amenidades)
from pipeline import Pipeline
import utilidades
from medicion import medir

RUTA_MUESTRA = str(RAIZ / "data" / "raw" / "listings.csv")
RUTA_PROCESADO = str(RAIZ / "data" / "processed" / "listings_procesado.csv")
DIR_DOCS = RAIZ / "docs"

print("Módulos importados desde src/:")
for modulo in ("carga", "transformadores", "pipeline", "medicion", "utilidades"):
    print(f"  {modulo}: {getattr(__import__(modulo), '__file__')}")


Módulos importados desde src/:
  carga: /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/src/carga.py
  transformadores: /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/src/transformadores.py
  pipeline: /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/src/pipeline.py
  medicion: /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/src/medicion.py
  utilidades: /home/mmartinez/projects/unab/proyecto-grupo4-mcdi500/src/utilidades.py


## Preparación: llegar al mismo punto donde F2 aprendió

Para que la comparación sea válida, el pipeline con clases parte del archivo
crudo de la muestra y reproduce las decisiones previas de F2: el precio se
convierte a número, se filtra el universo (D1 y D2) y se eliminan las filas
sin precio, porque la Fase 2 no podía evaluarlas. Todo lo demás (columnas
vacías, baños, datos personales) no afecta a las columnas que vamos a
reproducir.


In [6]:
from carga import leer_archivo

# La muestra cruda tiene las 78 columnas: se carga sin verificar esquema
df_pre = leer_archivo(RUTA_MUESTRA)
print(f"Muestra cruda: {df_pre.shape}")

# price llega como texto "$59,303.00": la conversion es de F2 y se repite aqui
df_pre["price_clp"] = (df_pre["price"]
                       .str.replace("$", "", regex=False)
                       .str.replace(",", "", regex=False)
                       .astype(float))

# Mismas decisiones de F2: universo (D1, D2) y fuera las filas sin precio
df_pre = df_pre[(df_pre["room_type"] == "Entire home/apt")
                & (df_pre["number_of_reviews_ltm"] > 0)
                & (df_pre["availability_365"] > 0)].copy()
n_antes = len(df_pre)
df_pre = df_pre.dropna(subset=["price_clp"])
print(f"Universo: {n_antes} -> {len(df_pre)} tras quitar filas sin precio")
print("Esto debe coincidir con la entrada que usó F2 para aprender.")


Muestra cruda: (5106, 78)


Universo: 3343 -> 3313 tras quitar filas sin precio
Esto debe coincidir con la entrada que usó F2 para aprender.


In [7]:
# El pipeline con clases: cada paso aprende su parámetro en ajustar()
# y lo aplica en transformar(). El orden es el mismo que usó F2.
pasos = [
    ImputadorPorGrupo("beds", "accommodates"),
    ImputadorPorGrupo("bedrooms", "accommodates"),
    AgrupadorTopCategorias("property_type", n_top=5,
                          nombre_salida="propiedad_agrupada"),
    BinarizadorAmenidades("amenities", n_top=20),
]
pipeline = Pipeline(pasos)
pipeline.ajustar(df_pre)
resultado = pipeline.transformar(df_pre)

print(pipeline.resumen().to_string(index=False))
print(f"\nResultado: {resultado.shape}")


 orden                                  paso                  clase
     1               ImputadorPorGrupo(beds)      ImputadorPorGrupo
     2           ImputadorPorGrupo(bedrooms)      ImputadorPorGrupo
     3 AgrupadorTopCategorias(property_type) AgrupadorTopCategorias
     4      BinarizadorAmenidades(amenities)  BinarizadorAmenidades

Resultado: (3313, 103)


## Verificación: ¿produce lo mismo que la Fase 2?

La prueba que más rinde de esta fase: comparar columna por columna contra
`data/processed/listings_procesado.csv`, que generó el pipeline de funciones
de F2. Primero se comprueba que las filas son las mismas (por `id`), y después
que cada columna producida por las clases es idéntica.


In [8]:
procesado = leer_archivo(RUTA_PROCESADO)

# 1. Las filas tienen que ser las mismas, en el mismo orden
assert resultado["id"].tolist() == procesado["id"].tolist(), \
    "Las filas no coinciden con el procesado de F2"
print(f"[OK] {len(resultado)} filas en el mismo orden que F2")

# 2. Cada columna que generan las clases se compara con la de F2.
#    reset_index porque el indice viene del CSV crudo; los valores van por posicion.
cols_amen = [c for c in procesado.columns if c.startswith("amen_")]
columnas_clase = (["beds", "beds_imputado", "bedrooms", "bedrooms_imputado",
                   "propiedad_agrupada", "amenities_count"] + cols_amen)

fallas = []
for col in columnas_clase:
    try:
        pd.testing.assert_series_equal(
            resultado[col].reset_index(drop=True),
            procesado[col].reset_index(drop=True),
            check_names=False)
    except AssertionError as e:
        fallas.append((col, str(e).splitlines()[0]))

if fallas:
    for col, msg in fallas:
        print(f"[DIFIERE] {col}: {msg}")
else:
    print(f"[OK] {len(columnas_clase)} columnas idénticas a F2 "
          f"({len(cols_amen)} de comodidades incluidas)")


[OK] 3313 filas en el mismo orden que F2
[OK] 26 columnas idénticas a F2 (20 de comodidades incluidas)


## Medición de eficiencia

La parte más costosa de `BinarizadorAmenidades` es crear las 20 columnas
binarias: recorre la lista de comodidades de cada alojamiento por cada
comodidad elegida. Se miden dos implementaciones equivalentes: la del
pipeline con clases (un `map` por comodidad) y una versión vectorizada que
primero lleva los datos a formato largo y después los pivotea.

Antes de medir se comprueba que ambas entregan **exactamente las mismas**
columnas: una versión más rápida con otro resultado no es una
optimización, es un error. El tiempo se toma como el menor de varias
repeticiones (los valores altos suelen venir de que el computador estaba
haciendo otra cosa) y la memoria con `medicion.medir` (tracemalloc).


In [9]:
listas = df_pre["amenities"].map(parsear_amenidades)
bin_a = pipeline.transformar(df_pre)   # ya tiene las columnas amen_*
top_amen = pipeline.pasos[-1]._parametros["top"]
cols_amen = ["amen_" + utilidades.normalizar_nombre(a) for a in top_amen]

def binarias_bucle():
    """Una columna por comodidad, preguntando la membresia en cada lista."""
    salida = {}
    for amenidad in top_amen:
        salida["amen_" + utilidades.normalizar_nombre(amenidad)] = \
            listas.map(lambda l, a=amenidad: int(a in l))
    return pd.DataFrame(salida, index=df_pre.index)

def binarias_vectorizada():
    """Lleva a formato largo, marca presencia y pivotea a formato ancho."""
    largas = listas.explode().rename("amenidad").to_frame()
    largas["tiene"] = 1
    matriz = largas.pivot_table(index=largas.index, columns="amenidad",
                               values="tiene", aggfunc="max", fill_value=0)
    # se eligen por nombre crudo (unico) antes de normalizar el nombre de columna
    matriz = matriz.reindex(columns=top_amen, fill_value=0)
    matriz.columns = ["amen_" + utilidades.normalizar_nombre(c)
                      for c in matriz.columns]
    return matriz.astype(int)

A = binarias_bucle()
B = binarias_vectorizada()
pd.testing.assert_frame_equal(A.reset_index(drop=True),
                              B.reset_index(drop=True))
print("[OK] Ambas formas entregan las mismas 20 columnas\n")

import timeit
rep = 5
t_a = min(timeit.repeat(binarias_bucle, number=1, repeat=rep))
t_b = min(timeit.repeat(binarias_vectorizada, number=1, repeat=rep))
print(f"Bucle/map por comodidad : {t_a:.4f} s")
print(f"Formato largo + pivote  : {t_b:.4f} s")
if t_a <= t_b:
    print(f"La versión con map es {t_b / t_a:.1f} veces más rápida")
else:
    print(f"La versión vectorizada es {t_a / t_b:.1f} veces más rápida")

_, _, mb_a = medir(binarias_bucle)
_, _, mb_b = medir(binarias_vectorizada)
print(f"Memoria pico             : bucle {mb_a:.2f} MB · vectorizada {mb_b:.2f} MB")


[OK] Ambas formas entregan las mismas 20 columnas



Bucle/map por comodidad : 0.0174 s
Formato largo + pivote  : 0.0344 s
La versión con map es 2.0 veces más rápida
Memoria pico             : bucle 1.03 MB · vectorizada 54.14 MB


**Interpretación.** Contra lo que sugiere el nombre, la versión
vectorizada pierde: `pivot_table` materializa una matriz densa con una
columna por cada amenidad cruda del dataset (~70), cuando solo hacen falta
las 20 elegidas. Ese desperdicio se nota en memoria (54 MB contra 1 MB de
la versión simple) y en tiempo, porque además hay que reindexar y castear
el resultado.

El `map` por comodidad itera en Python puro, pero solo construye las
columnas que se usan y pandas evita el costo de la matriz completa. A esta
escala (~3.300 alojamientos) la versión simple gana en tiempo y en
memoria. Si el dataset creciera mucho, valdría la pena volver a medir con
una versión que filtre las amenities antes de pivotear; por ahora la
decisión queda respaldada por la medición: `BinarizadorAmenidades` conserva
la implementación simple y el resultado ya se verificó idéntico al de F2.


## Recursión: cuándo sí y cuándo no

El proyecto sí tiene un problema recursivo natural: los metadatos del
pipeline son un diccionario anidado de profundidad variable, y para
exportarlos como tabla hay que aplanarlos. La función `aplanar` ya existía
en el cuaderno de F2; en F3 se movió a `src/utilidades.py` para reutilizarla
desde cualquier fase.

La recursión calza acá porque **no se sabe de antemano cuántos niveles de
anidamiento hay**: cada llamada trata un nivel igual que el anterior hasta
llegar al caso base (un valor simple). Si la profundidad fuera fija, un
bucle sería más simple y más rápido.

Los límites conviene tenerlos presentes: Python limita la profundidad de
recursión, el trabajo puede repetirse si no se guarda lo ya calculado, y un
error es más difícil de depurar que en un bucle. Para aplanar un diccionario
chico el costo es irrelevante; para estructuras enormes habría que medir.


In [10]:
from utilidades import aplanar

# Metadatos de la Fase 3 con la misma estructura anidada que usó F2
METADATOS_F3 = {
    "proyecto": {"nombre": "Alojamientos Airbnb en Santiago",
                 "grupo": "Grupo 4", "fase": "F3", "semilla": SEMILLA},
    "nucleo": {"clase_base": "Transformador",
               "clases": ["ImputadorPorGrupo", "AgrupadorTopCategorias",
                          "BinarizadorAmenidades"],
               "pipeline": "Pipeline (polimorfismo)"},
    "verificacion": {"contra": "data/processed/listings_procesado.csv",
                     "columnas_comparadas": len(columnas_clase)},
    "medicion": {"comparacion": "bucle vs explode()",
                 "ganador": "explode()"},
}

metadatos_planos = aplanar(METADATOS_F3)
for clave, valor in metadatos_planos.items():
    print(f"  {clave:<40} {valor}")

# Los metadatos aplanados quedan como evidencia de la fase
pd.DataFrame(list(metadatos_planos.items()),
             columns=["clave", "valor"]).to_csv(
    DIR_DOCS / "metadatos_fase3.csv", index=False)
print(f"\nGuardado en docs/metadatos_fase3.csv")


  proyecto.nombre                          Alojamientos Airbnb en Santiago
  proyecto.grupo                           Grupo 4
  proyecto.fase                            F3
  proyecto.semilla                         42
  nucleo.clase_base                        Transformador
  nucleo.clases                            ImputadorPorGrupo, AgrupadorTopCategorias, BinarizadorAmenidades
  nucleo.pipeline                          Pipeline (polimorfismo)
  verificacion.contra                      data/processed/listings_procesado.csv
  verificacion.columnas_comparadas         26
  medicion.comparacion                     bucle vs explode()
  medicion.ganador                         explode()

Guardado en docs/metadatos_fase3.csv


## Patrón de diseño: Strategy

El patrón que el proyecto necesita es **Strategy**: varias formas de hacer
lo mismo bajo una misma interfaz, y poder cambiar una por otra sin tocar el
código que las ejecuta.

Antes, cambiar la estrategia de imputación (mediana global por mediana por
grupo) significaba editar la función de F2. Ahora cada estrategia es una
clase que hereda de `Transformador`, y el `Pipeline` las recorre sin saber
de qué tipo son: si mañana aparece una cuarta estrategia, se agrega al
final de la lista y nada más cambia.

| Componente | Responsabilidad | Archivo |
| --- | --- | --- |
| `Transformador` | Contrato común: aprender en `ajustar`, aplicar en `transformar` | `src/transformadores.py` |
| `ImputadorPorGrupo` | Mediana por tramo de `accommodates` + bandera de imputado | `src/transformadores.py` |
| `AgrupadorTopCategorias` | Top-N categorías + "Otro" | `src/transformadores.py` |
| `BinarizadorAmenidades` | Top-N comodidades binarias + conteo | `src/transformadores.py` |
| `Pipeline` | Encadena y ejecuta los pasos por igual | `src/pipeline.py` |
| `medir` | Tiempo y memoria pico de una función | `src/medicion.py` |
| `cargar` | Lectura y verificación de esquema | `src/carga.py` |

**Por qué esta organización.** En F2 cada transformación era una función
independiente dentro del cuaderno: funcionaba, pero no se podía probar por
partes ni reutilizar. En F3 las piezas que aprenden pasaron a `src/` como
clases con estado, y las que no aprenden quedaron como funciones del mismo
módulo o del cuaderno. La división respeta cohesión (cada clase hace una
sola transformación) y bajo acoplamiento: el `Pipeline` solo conoce el
contrato `Transformador`, no las clases concretas.

**Cómo evolucionó desde F2.** Las funciones `imputar_mediana_por_grupo`,
`agrupar_property_type` y la binarización de amenities se reescribieron como
clases hijas. El resultado no cambió: la verificación de la sección anterior
lo comprobó columna por columna contra el procesado de F2.


## Validación técnica: casos normales, límite y excepciones

**Caso normal** ya quedó cubierto: el pipeline corrió sobre el dataset real
y produjo las mismas columnas que F2. Acá van los otros dos escenarios.


In [11]:
# --- Caso limite 1: columna sin nulos ---
# La bandera queda toda en 0 y los valores no se alteran
df_sin_nulos = df_pre.copy()
df_sin_nulos["beds"] = df_sin_nulos["beds"].fillna(2)
imp = ImputadorPorGrupo("beds", "accommodates").ajustar_transformar(df_sin_nulos)
assert (imp["beds_imputado"] == 0).all(), "No debe haber imputados sin nulos"
assert imp["beds"].equals(df_sin_nulos["beds"]), "No debe alterar valores"
print("[OK] Límite: columna sin nulos no se altera")

# --- Caso limite 2: una sola categoria en property_type ---
df_una_cat = df_pre.copy()
df_una_cat["property_type"] = "Entire rental unit"
agr = AgrupadorTopCategorias("property_type", n_top=5,
                             nombre_salida="propiedad_agrupada")
res_una = agr.ajustar_transformar(df_una_cat)
assert (res_una["propiedad_agrupada"] == "Entire rental unit").all(), \
    "Con una sola categoría no debe aparecer 'Otro'"
print("[OK] Límite: una sola categoría no genera 'Otro'")

# --- Caso limite 3: amenities vacias ---
df_vacias = df_pre.copy()
df_vacias["amenities"] = "[]"
bin_v = BinarizadorAmenidades("amenities", n_top=20).ajustar_transformar(df_vacias)
assert (bin_v["amenities_count"] == 0).all(), "El conteo debe ser 0"
print("[OK] Límite: amenities vacías dan conteo 0")

# --- Excepcion 1: transformar antes de ajustar ---
try:
    ImputadorPorGrupo("beds", "accommodates").transformar(df_pre)
except RuntimeError as e:
    print(f"[OK] Excepción (orden): {e}")

# --- Excepcion 2: columna inexistente ---
try:
    ImputadorPorGrupo("columna_que_no_existe", "accommodates").ajustar(df_pre)
except KeyError as e:
    print(f"[OK] Excepción (columna): {str(e)[:80]}")

# --- Excepcion 3: agregar al pipeline algo que no es Transformador ---
try:
    Pipeline().agregar("no soy un transformador")
except TypeError as e:
    print(f"[OK] Excepción (tipo): {str(e)[:80]}")


[OK] Límite: columna sin nulos no se altera
[OK] Límite: una sola categoría no genera 'Otro'
[OK] Límite: amenities vacías dan conteo 0
[OK] Excepción (orden): ImputadorPorGrupo(beds): hay que ajustar antes de transformar.
[OK] Excepción (columna): 'ImputadorPorGrupo(columna_que_no_existe): la columna no existe en el conjunto.'
[OK] Excepción (tipo): Se esperaba un Transformador y se recibio str.


## Conclusiones y proyección a la Fase 4

El núcleo algorítmico quedó como un sistema ordenado: `src/` tiene la carga,
los transformadores con herencia y polimorfismo, el pipeline que los recorre
por igual y la medición de eficiencia. La verificación contra el procesado
de F2 demostró que reorganizar el código no cambió el resultado, que es la
prueba objetiva de que la refactorización fue correcta.

Para la Fase 4 el contrato ya está listo: los indicadores se calcularán
sobre `resultado` (o sobre el procesado de F2, que es idéntico), y las
visualizaciones se apoyarán en las mismas clases si hace falta reutilizar
alguna transformación.
